# Stable Diffusion Deep Dive

Original: https://github.com/fastai/diffusion-nbs/

Adapted: Antonio Esteves, UMinho

* * *
Stable Diffusion is a powerful text-to-image acessible through the [Huggingface diffusers library](https://huggingface.co/blog/stable_diffusion), where generating images can be as simple as:

```python
from diffusers import StableDiffusionPipeline
pipe  = StableDiffusionPipeline.from_pretrained(
  "CompVis/stable-diffusion-v1-4",
  variant="fp16",
  torch_dtype=torch.float16, use_auth_token=True
).to("cuda")
image = pipe("An astronaught scuba diving").images[0]
```

In this notebook we are going to dig into the code behind these easy-to-use interfaces provide by the `diffusers` library, to understand how they work under the hood. We will recreate the functionality behind the pipipeline used in the previous code, and then one by one we will inspect the different components to figure out what they do.

## Setup and Necessary Libraries

To run the notebook, it is necessary to log into huggingface and accept the terms of the licence associated with  the Stable Diffusion model we are going to use. See the [model card](https://huggingface.co/CompVis/stable-diffusion-v1-4) for the details. When we run this notebook for the first time, it is necessary to uncomment the following two cells to install the requirements and log in to huggingface with an access token.

In [ ]:
!pip install -q --upgrade transformers==4.25.1 diffusers ftfy accelerate

In [ ]:
from   base64 import b64encode

import numpy
import torch
from   diffusers       import AutoencoderKL, LMSDiscreteScheduler, UNet2DConditionModel
from   huggingface_hub import notebook_login

# For image display
from   IPython.display import HTML
from   matplotlib      import pyplot as plt
from   pathlib         import Path
from   PIL             import Image
from   torch           import autocast
from   torchvision     import transforms as tfms
from   tqdm.auto       import tqdm
from   transformers    import CLIPTextModel, CLIPTokenizer, logging
import os

torch.manual_seed(1)
if not (Path.home()/'.cache/huggingface'/'token').exists(): notebook_login()

# Supress some unnecessary warnings when loading the CLIPTextModel
logging.set_verbosity_error()

# Set the computing device
torch_device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
if "mps" == torch_device: os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = "1"

print(f'Computing device is {torch_device}')

## Loading the models

The code in this section and the next one comes from the [Huggingface example notebook](https://colab.research.google.com/github/huggingface/notebooks/blob/main/diffusers/stable_diffusion.ipynb). It downloads and sets up the relevant models and components we will be using.

If we instanciated a pipeline as explained before, we can access the pipeline's components using `pipe.unet`, `pipe.vae`, and so on.

In this notebook we are NOT going to apply any memory-saving tricks, so if we run into out of GPU RAM, look at the pipeline code for inspiration with things like attention slicing, switching to half precision (fp16), keeping the VAE on the CPU, and other modifications.

In [ ]:
# Load the autoencoder model which will be used to decode the latents into image space
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae")

# Load the tokenizer and text encoder to tokenize and encode the text
tokenizer    = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14")

# The UNet model for denoising the latents
unet = UNet2DConditionModel.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="unet")

# The noise scheduler
scheduler = LMSDiscreteScheduler(beta_start=0.00085, beta_end=0.012, beta_schedule="scaled_linear", num_train_timesteps=1000)

# Place the models in the computing device
vae          = vae.to(torch_device)
text_encoder = text_encoder.to(torch_device)
unet         = unet.to(torch_device);

## A diffusion loop

What we want to do in this notebook is dig a little deeper into how sampling with the stable diffusion works, so we will start by checking that the example code runs. Again, this is adapted from the [HF notebook](https://colab.research.google.com/github/huggingface/notebooks/blob/main/diffusers/stable_diffusion.ipynb) and is very similar the [`__call__()` method of the stable diffusion pipeline](https://github.com/huggingface/diffusers/blob/main/src/diffusers/pipelines/stable_diffusion/pipeline_stable_diffusion.py#L200).  

In [ ]:
# Hyperparameters
prompt              = ["A watercolor painting of an otter"]
height              = 512                   # default image height in Stable Diffusion
width               = 512                   # default image width  in Stable Diffusion
num_inference_steps = 30                    # number of denoising steps
guidance_scale      = 7.5                   # scale applied in classifier-free guidance
generator           = torch.manual_seed(32) # seed generator to create the initial latent noise
batch_size          = 1

# Convert the conditioning text prompt into tokens using the CLIP tokenizer
text_input = tokenizer(
  prompt,
  padding        = "max_length",
  max_length     = tokenizer.model_max_length,
  truncation     = True,
  return_tensors = "pt"
)
# Obtain the conditioning text prompt embeddings using the CLIP text encoder
with torch.no_grad():
    text_embeddings = text_encoder(text_input.input_ids.to(torch_device))[0]

# Convert an "empty" unconditioning text prompt into tokens using
# the CLIP tokenizer (this is necessary in classifier free guidance - CFG)
max_length   = text_input.input_ids.shape[-1]
uncond_input = tokenizer(
    [""] * batch_size,
    padding        = "max_length",
    max_length     = max_length,
    return_tensors = "pt"
)

# Obtain the unconditioning text prompt embeddings using the CLIP text encoder
with torch.no_grad():
    uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))[0]

# Concatenate conditioning and unconditioning text prompt embeddings
text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

# Prepare the noise scheduler
def set_timesteps(scheduler, num_inference_steps):
    scheduler.set_timesteps(num_inference_steps)
    # minor fix to ensure MPS compatibility (fixed in diffusers PR 3925)
    scheduler.timesteps = scheduler.timesteps.to(torch.float32)

set_timesteps(scheduler, num_inference_steps)

# Prepare the initial latents of the denoising process
latents = torch.randn(
  (batch_size, unet.in_channels, height // 8, width // 8),
  generator=generator,
)
latents = latents.to(torch_device)
latents = latents * scheduler.init_noise_sigma # Scale the latent using the noise variance

# ...............................................................................
# Sampling/denoising loop to generate an image conditioned by the provided prompt
# ...............................................................................

with autocast("cuda"):  # will fallback to CPU if no CUDA
    for i, t in tqdm(enumerate(scheduler.timesteps), total=len(scheduler.timesteps)):

        # Duplicate the latents if we are applying classifier-free guidance
        # to avoid executing two forward passes
        latent_model_input = torch.cat([latents] * 2)

        # Scale the latents (preconditioning)
        latent_model_input   = scheduler.scale_model_input(latent_model_input, t)

        # predict the noise that needs to be removed from the latents
        with torch.no_grad():
            noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embeddings).sample

        # Apply CFG guidance
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        # Obtain the previous noisy sample: given z_t predict z_{t-1}
        latents = scheduler.step(noise_pred, t, latents).prev_sample

# scale and decode the final latent (z_0) into the image spcae (x_0) with the VAE
latents = 1 / 0.18215 * latents
with torch.no_grad():
    image = vae.decode(latents).sample

# Display the image
image      = (image / 2 + 0.5).clamp(0, 1)
image      = image.detach().cpu().permute(0, 2, 3, 1).numpy()
images     = (image * 255).round().astype("uint8")
pil_images = [Image.fromarray(image) for image in images]
pil_images[0]

The sampling pipeline is working. Let us now look at the pipeline's components one by one.

## The Autoencoder (AE)

The AE can 'encode' an image into some sort of latent representation, and decode it back into an image. The AE code is organized in a couple of functions.

In [ ]:
'''
Encode a single image into a single latent, with shape [1, 4, 64, 64]
using the VAE encoder.
'''
def encode_image_to_latent(input_img):
    # Single image -> single latent of shape (1, 4, 64, 64)
    with torch.no_grad():
        latent = vae.encode(tfms.ToTensor()(input_img).unsqueeze(0).to(torch_device)*2-1) # Note scaling
    return 0.18215 * latent.latent_dist.sample()

'''
Decode a batch of latents into images, using the VAE decoder.
'''
def decode_latents_to_images(latents):
    # bath of latents -> list of images
    latents = (1 / 0.18215) * latents
    with torch.no_grad():
        images_t = vae.decode(latents).sample
    images_t   = (images_t / 2 + 0.5).clamp(0, 1)
    images_t   = images_t.detach().cpu().permute(0, 2, 3, 1).numpy()
    images     = (images_t * 255).round().astype("uint8")
    pil_images = [Image.fromarray(image) for image in images]
    return pil_images

We will use a picture from the Web here, but we can load our own instead by uploading it and editing the filename in the next cell.

In [ ]:
# Download a demo Image
!curl --output macaw.jpg 'https://lafeber.com/pet-birds/wp-content/uploads/2018/06/Scarlet-Macaw-2.jpg'

In [ ]:
# Show the image with PIL
input_image = Image.open('macaw.jpg').resize((512, 512))
input_image

Now, we encode the image into the latent space of the AE with the function defined above.

In [ ]:
# Encode the selected image into the latent space
encoded = encode_image_to_latent(input_image)
print(f'Latent shape: {encoded.shape}')

In [ ]:
# Let us visualize as grey images the four channels of the latent representation
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for c in range(encoded.shape[1]):
    axs[c].imshow(encoded[0][c].cpu(), cmap='Greys')

This 4x64x64 tensor captures lots of information about the image, hopefully enough that when we feed it through the decoder we get back something very close to our input image.

In [ ]:
# Decode the latent representation back into an image
decoded = decode_latents_to_images(encoded)[0]
decoded

It is possible to observe small differences between input image and the reconstructed image, for example on the bird's eye. This is pretty impressive, a 4x64x64 latent seems to hold a lot more information than a 64x64 image.

The autoencoder we are using was trained to compress an image to a smaller representation and then re-create the image back from this compressed version again.

In this particular case, the compression factor is 48, we start with a 3x512x512(CxHxW) image and it is compressed into a latent vector of shape 4x64x64. Each 3x8x8 pixel volume in the input image gets compressed down to just 4 numbers(4x1x1). You can find AEs with a higher compression ratio. For example, some popular VQGAN models apply a 16X compression ration, but at some point they begin to introduce artifacts that we do not want.

Why do we even use an autoencoder? We can do diffusion in pixel space, where the model gets all the image data as inputs and produces an output prediction of the same shape. But this means processing a lot of data, and make high-resolution generation very computationally expensive. Some solutions to this involve doing diffusion at low resolution, for example 64x64, and then training a separate model to upscale repeatedly, as occurs in DALL-E 2 and Imagen. Instead, latent diffusion does the diffusion process in the latent space, using the compressed representations from the AE rather than raw images. These representations are information rich, and can be small enough to handle manageably on consumer hardware. Once we have denoised a new latent representation, the AE decoder can convert the denoised final latent into pixels.

# The Scheduler

Now we discuss the noise scheduler. During training, we add noise to an image (forward pass) an then use the U-Net model to predict the added noise. If we always add a large amount of noise, the model might not have much to work with. If we only add a tiny amount of noise, the model would not be able to do much with the random starting point we use for sampling. So, during training, the amount of added noise is varied, according to some distribution.

During sampling, we want to denoise the initial latent over a number of steps. How many steps and how much noise we should remove at each step will affect the final result.

The scheduler is in charge of handling all of these details. For example, the `scheduler = LMSDiscreteScheduler(beta_start=0.00085, beta_end=0.012, beta_schedule="scaled_linear", num_train_timesteps=1000)` sets up a scheduler that matches the one used to train the U-Net model. When we want to sample over a smaller number of steps, we set the `scheduler.set_timesteps` as follows.

In [ ]:
# Setting the number of sampling steps with a small value (15)
set_timesteps(scheduler, 15)

You can inspect the values of the times that correspond to the schedule we just defined, relative to the 1000 steps used during training.

In [ ]:
# Print the 15 times of our sampling schedule relative
# to the original 1000 steps used during training of the U-Net
print(scheduler.timesteps)

We can also inspect the amount of noise (variance) associated to each sampling time.

In [ ]:
# Print the noise level associated with the 15 sampling times
print(scheduler.sigmas)

During sampling, we start at a high noise level (in fact, our input latent is pure noise) and gradually denoise the noisy latent into a "clean" latent, according to this noise variance schedule.

In [ ]:
# Plot the 15-values noise schedule
plt.plot(scheduler.sigmas)
plt.title('Noise schedule')
plt.xlabel('sampling step')
plt.ylabel('noise variance')
plt.show()

In [ ]:
# Plot the 15 times of the noise schedule
plt.plot(scheduler.timesteps)
plt.title('Noise schedule')
plt.xlabel('sampling step')
plt.ylabel('denoising timestep')
plt.show()

## A Denoising Step

The `scheduler.sigmas` is the amount of noise added to the latent representation in the different steps. Let us visualize what this looks like by adding a bit of noise to our encoded image and then decode the noisy latent.

In [ ]:
noise              = torch.randn_like(encoded) # Random noise
sampling_step      = 10                        # Equivalent to step 10 out of 15 in our schedule
encoded_and_noised = scheduler.add_noise(
  encoded,
  noise,
  timesteps = torch.tensor([scheduler.timesteps[sampling_step]])
)
decode_latents_to_images(encoded_and_noised.float())[0] # Display the reconstructed image

To see the result of different timesteps, we can repeat the previous cell using different values for `sampling_step` in the range 0..14.

If we uncomment the cell below, we will see that the `scheduler.add_noise` function literally just adds noise scaled by the noise variance according to the expression: `noisy_samples = original_samples + noise * sigmas`

In [ ]:
scheduler.add_noise

Other diffusion models may be trained with different noising and scheduling approaches, some of which keep the variance fairly constant across noise levels (variance preserving) with different scaling and mixing tricks instead of having noisy latents with higher and higher variance as more noise is added (variance exploding).

If we want to start from random noise instead of a noised image, we need to scale it by the largest sigma value used during training, ~14 in our case. And before these noisy latents are fed into the model they are scaled again in the so-called pre-conditioning step,
`latent_model_input = latent_model_input / ((sigma**2 + 1) ** 0.5)`, which is handled by the operation `latent_model_input = scheduler.scale_model_input(latent_model_input, t)`.

Again, this scaling/pre-conditioning differs between papers and implementations of diffusion models.

## Sampling starting from a noisy latent instead of pure noise (image2image)

Let us see what happens when we use an image as a starting point, adding some noise to it and then doing the denoising steps in a loop with a new prompt.

We will use a loop similar to the one presented before, but we will skip the first `start_step` steps.

For adding noise to our image, we will use code similar to that shown above, using the scheduler to add noise with a level correspondent to the selected step, `start_step=10` in our case.

In [ ]:
# The hyperparameters are the same as before except for the new prompt
prompt              = ["A colorful dancer, nat geo photo"]
height              = 512                   # default image height in Stable Diffusion
width               = 512                   # default image width in Stable Diffusion
num_inference_steps = 50                    # number of denoising steps
guidance_scale      = 8                     # scale applied in classifier-free guidance
generator           = torch.manual_seed(32) # seed generator to create the initial latent noise
batch_size          = 1

# Convert the conditioning text prompt into tokens using the CLIP tokenizer
text_input = tokenizer(
  prompt,
  padding        = "max_length",
  max_length     = tokenizer.model_max_length,
  truncation     = True,
  return_tensors = "pt"
)

# Obtain the conditioning text prompt embeddings using the CLIP text encoder
with torch.no_grad():
    text_embeddings = text_encoder(text_input.input_ids.to(torch_device))[0]

# Convert an "empty" unconditioning text prompt into tokens using
# the CLIP tokenizer (this is necessary in classifier free guidance - CFG)
max_length   = text_input.input_ids.shape[-1]
uncond_input = tokenizer(
  [""] * batch_size,
  padding        = "max_length",
  max_length     = max_length,
  return_tensors = "pt"
)
# Obtain the unconditioning text prompt embeddings using the CLIP text encoder
with torch.no_grad():
    uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))[0]

# Concatenate conditioning and unconditioning text prompt embeddings
text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

# Prepare the noise scheduler (setting the number of inference steps)
set_timesteps(scheduler, num_inference_steps)

# Prepare the initial latents of the denoising process,
# adding noise appropriate for 'start_step'
start_step  = 20
start_sigma = scheduler.sigmas[start_step]
noise       = torch.randn_like(encoded)
latents     = scheduler.add_noise(
  encoded,
  noise,
  timesteps=torch.tensor([scheduler.timesteps[start_step]])
)
latents     = latents.to(torch_device).float()

# ...............................................................................
# Sampling/denoising loop to generate an image conditioned by the provided prompt
# ...............................................................................

for i, t in tqdm(enumerate(scheduler.timesteps), total=len(scheduler.timesteps)):

    if i >= start_step: # This is the only modification to the previous sampling loop

        # Duplicate the latents if we are applying classifier-free guidance
        # to avoid executing two forward passes
        latent_model_input = torch.cat([latents] * 2)

        # Scale the latents (preconditioning)
        sigma              = scheduler.sigmas[i]
        latent_model_input = scheduler.scale_model_input(latent_model_input, t)

        # predict the noise that needs to be removed from the latents
        with torch.no_grad():
            noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embeddings)["sample"]

        # Apply CFG guidance
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        # Obtain the previous noisy sample: given z_t predict z_{t-1}
        latents = scheduler.step(noise_pred, t, latents).prev_sample

# Decode the final latent (z_0) into the image spcae (x_0) with the VAE
decode_latents_to_images(latents)[0]

We can see that some colours and structure from the original image are preserved, but we now have a new picture. The more noise we add and the more steps we run, the further away it gets from the input image. This is how the popular img2img pipeline works. Again, if this is our end goal there are tools to make this easy! But we can see that under the hood this is the same as the generation loop just skipping the first few steps and starting from a noised image rather than pure noise.

Explore changing how many steps are skipped and see how this affects the amount the image changes from the input. As the starting step gets higher, meaning that we skip more denoising steps, the final images becomes more and more similar to the initial image, meaning it changed less.

## Exploring the text embedding pipeline

We use a text encoder model to turn our text into a set of embeddings, which are fed to the diffusion model as conditioning. Let us follow a fragment of text through this process and see how it works.

In [ ]:
# The text prompt
prompt = 'A picture of a puppy'

We begin with tokenization:

In [ ]:
# Turn the text into a sequence of tokens
text_input = tokenizer(
    prompt,
    padding        = "max_length",
    max_length     = tokenizer.model_max_length,
    truncation     = True,
    return_tensors = "pt"
)
text_input['input_ids'][0] # View the tokens

In [ ]:
# See the individual tokens using the tokenizer model 'decoder'.
# We only look at the first 8 tokens, since after position 6 all tokens are '<|endoftext|>'.
for t in text_input['input_ids'][0][:8]: 
    print(t, tokenizer.decoder.get(int(t)))

Note that 6829 is "puppy".

We now get the token embeddings using the CLIP text encoder.

In [ ]:
# Obtain the text embeddings ***** CELL_OUT_EMBEDDINGS *****
output_embeddings = text_encoder(text_input.input_ids.to(torch_device))[0]
print('Embeddings shape:', output_embeddings.shape)
output_embeddings

We pass our tokens through the `text_encoder` and get some numbers we can feed into the model.

How are these numbers generated? The tokens are transformed into a set of input embeddings, which are then fed into a transformer model to get the final output embeddings.

To get these input embeddings, the input tokens go through two steps, as revealed by inspecting the implementation of `text_encoder.text_model.embeddings`: a token `Embedding`layer and a position `Embedding` layer.

In [ ]:
text_encoder.text_model.embeddings

### Token embeddings

The token is fed to the `token_embedding` to transform it into a vector. The name `get_input_embeddings` of the function that we will use later on is a bit inadequate since the token embeddings need to be combined with the position embeddings before they are actually the input embeddings we will apply into the model. Anyway, let us look at just the token embedding layer first.

In [ ]:
# Access the token embedding layer
token_emb_layer = text_encoder.text_model.embeddings.token_embedding
print(token_emb_layer) 
# Embedding(49408, 768) => vocabulary size of 49408 and embedding dimension of  768

We will now embed a token: 6829, corresponding to "puppy".

In [ ]:
# Embed a token, in this case the one for "puppy".
embedding = token_emb_layer(torch.tensor(6829, device=torch_device))
print(f'Token embedding shape: {embedding.shape}') # a vector of size 768

The single token has been mapped to a 768-dimensional vector, the token embedding.

We can do the same with all of the tokens in the prompt to get all the token embeddings.

In [ ]:
token_embeddings = token_emb_layer(text_input.input_ids.to(torch_device))
print(f'Token embeddings shape: {token_embeddings.shape}') # batch size 1, 77 tokens, 768 values per token
print(f'Token embeddings:\n{token_embeddings}')

### Positional Embeddings

The positional embeddings tell the model where in a sequence a token is. Much like the token embedding, this is a set of (optionally learnable) parameters. But now instead of dealing with ~50k tokens we just need one for each position, a total of 77.

In [ ]:
pos_emb_layer = text_encoder.text_model.embeddings.position_embedding
print(pos_emb_layer)

We can get the position embedding for each token.

In [ ]:
position_ids        = text_encoder.text_model.embeddings.position_ids[:, :77]
position_embeddings = pos_emb_layer(position_ids)
print(f'Position embeddings shape: {position_embeddings.shape}')
print(f'Position embeddings:\n{position_embeddings}')

### Combining token and position embeddings

It is time to combine token and position embeddings. We just add them but other approaches are possible. For our model, this is how it is done. Combining the embeddings in this way gives us the final input embeddings ready to feed through the transformer model.

In [ ]:
# Combinine token and position embeddings to get the input embeddings
input_embeddings = token_embeddings + position_embeddings
print(f'Input embeddings shape: {input_embeddings.shape}')
print(f'Input embeddings:\n{input_embeddings}')

We can check that these values are the same we would get using `text_encoder.text_model.embeddings()`.

In [ ]:
# The following line combines the 3 cells above, but does not let us see the intermediate steps
text_encoder.text_model.embeddings(text_input.input_ids.to(torch_device))

### Feed the input embedding into the transformer model

![transformer diagram](https://github.com/johnowhitaker/tglcourse/raw/main/images/text_encoder_noborder.png)

We want to mess with the input embeddings, specifically the token embeddings, before we send them through the transformer, but first we should check that we know how to do that. To be sure, we must read the implementation of the `text_encoder`'s `forward` method, that is based on the `forward` method of the `text_model` that the `text_encoder` wraps. To inspect this method, we can type `??text_encoder.text_model.forward` in a cell and we will get the function information and some source code.

In [ ]:
??text_encoder.text_model.forward

Based on this documentation, we can copy what is necessary to get the 'last hidden state' and thus generate our final embeddings.

In [ ]:
def build_causal_attention_mask(bsz, seq_len, dtype):
    mask = torch.empty(bsz, seq_len, seq_len, dtype=dtype)
    mask.fill_(torch.tensor(torch.finfo(dtype).min))  # fill with large negative number (acts like -inf)
    mask = mask.triu_(1)  # zero out the lower diagonal to enforce causality
    return mask.unsqueeze(1)  # add a batch dimension

# Update your function call to use the new mask function
def get_output_embeds(input_embeddings):
    bsz, seq_len          = input_embeddings.shape[:2]
    causal_attention_mask = build_causal_attention_mask(bsz, seq_len, dtype=input_embeddings.dtype)
    
    # Getting the output embeddings involves calling the model with passing output_hidden_states=True
    # so that it does not just return the pooled final predictions
    encoder_outputs = text_encoder.text_model.encoder(
        inputs_embeds=input_embeddings,
        attention_mask=None, # We are not using an attention mask so this argument can be 'None'
        causal_attention_mask=causal_attention_mask.to(torch_device),
        output_attentions=None,
        output_hidden_states=True, # We want the output embeddings, not the final output
    )

    # We are interested in the output hidden state only (the first object in the returned tuple)
    output = encoder_outputs[0]

    # There is a final layer norm and we have to pass the embeddings through it
    output = text_encoder.text_model.final_layer_norm(output)

    # And we now have the output embeddings
    return output

out_embs_test = get_output_embeds(input_embeddings)      # Feed through the model with our new function
print(f'Output embeddings shape: {out_embs_test.shape}') # Check the output embeddings shape
print(f'Output embeddings:\n{out_embs_test}')            # Inspect the output embeddings

Note that these embeddings match the `output_embeddings` we saw near the cell marked as **"CELL_OUT_EMBEDDINGS"** and as so we have identified how to split the task "get the text embeddings" into multiple sub-steps that we can modify.

Now that we have this process in place, we can replace the input embedding of a token with a new one of our choice, which in our final use case will be something we want to learn. To demonstrate the concept though, let us replace the input embedding for 'puppy', in the prompt we have been playing with, with the embedding for token 2368, get a new set of output embeddings based on this, and use these to generate an image to see what we get.

In [ ]:
prompt = 'A picture of a puppy'

# Tokenize
text_input = tokenizer(
    prompt,
    padding="max_length", 
    max_length=tokenizer.model_max_length, 
    truncation=True, 
    return_tensors="pt"
)
input_ids  = text_input.input_ids.to(torch_device)

# Get token embeddings
token_embeddings = token_emb_layer(input_ids)

# The embedding for a new token = 2368
replacement_token_embedding = text_encoder.get_input_embeddings()(torch.tensor(2368, device=torch_device))

# Insert the new embedding into the token embeddings
token_embeddings[0, torch.where(input_ids[0]==6829)] = replacement_token_embedding.to(torch_device)

# Combine token embeddings with the position embeddings
input_embeddings = token_embeddings + position_embeddings

#  Feed the input embeddings through the transformer to get the output embeddings
modified_output_embeddings = get_output_embeds(input_embeddings)

print(f'Modified output embeddings shape: {modified_output_embeddings.shape}') # Check the output embeddings shape
print(f'Modified output embeddings:\n{modified_output_embeddings}')            # Inspect the output embeddings

The first few embeddings are the same as before, the last ones do not. Everything at and after the position of the token we replaced are affected.

If all went well, we should see something other than a puppy when we use these to generate an image. And sure enough, we do!

In [ ]:
# Generate an image with the modified embeddings

def generate_with_embs(text_embeddings):
    height              = 512     # default image height in Stable Diffusion
    width               = 512     # default image width in Stable Diffusion
    num_inference_steps = 30      # number of denoising steps
    guidance_scale      = 7.5     # scale applied in classifier-free guidance
    generator           = torch.manual_seed(32) # seed generator to create the inital latent noise
    batch_size          = 1

    max_length   = text_input.input_ids.shape[-1]

    # Convert an "empty" unconditioning text prompt into tokens using 
    # the CLIP tokenizer (this is necessary in classifier free guidance - CFG)
    uncond_input = tokenizer(
        [""] * batch_size,
        padding        = "max_length",
        max_length     = max_length,
        return_tensors = "pt",
    )
    # Obtain the unconditioning text prompt embeddings using the CLIP text encoder
    with torch.no_grad():
        uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))[0]

    # Concatenate conditioning and unconditioning text prompt embeddings
    text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

    # Prepare the noise scheduler
    set_timesteps(scheduler, num_inference_steps)

    # Prepare the initial latents of the denoising process
    latents = torch.randn(
    (batch_size, unet.in_channels, height // 8, width // 8),
    generator = generator,
    )
    latents = latents.to(torch_device)
    latents = latents * scheduler.init_noise_sigma

    # ...............................................................................
    # Sampling/denoising loop to generate an image conditioned by the provided prompt
    # ...............................................................................
    
    for i, t in tqdm(enumerate(scheduler.timesteps), total=len(scheduler.timesteps)):
        # Duplicate the latents if we are applying classifier-free guidance 
        # to avoid executing two forward passes
        latent_model_input = torch.cat([latents] * 2)
        # Scale the latents (preconditioning)
        sigma              = scheduler.sigmas[i]
        latent_model_input = scheduler.scale_model_input(latent_model_input, t)

        # predict the noise that needs to be removed from the latents
        with torch.no_grad():
            noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embeddings)["sample"]

        # Apply CFG guidance
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        # Obtain the previous noisy sample: given z_t predict z_{t-1}
        latents = scheduler.step(noise_pred, t, latents).prev_sample

    return decode_latents_to_images(latents)[0]


In [ ]:
generate_with_embs(modified_output_embeddings)

So, we now know what token 2368 corresponds to "cat".

We will see a more compelling use case of our embeddings manipulation later, but the idea is that once we can access and modify the token embedding we can do tricks like replacing them with something else. In the example we just did, that was just another token embedding from the model's vocabulary, equivalent to just editing the prompt. But we can also mix tokens - for example, here is a half puppy-half skunk example.

In [ ]:
# To find the token(s) for a word, like "skunk", or the embeddings for that token(s) we can use:
prompt = 'skunk'
print(f'"skunk" tokens: {tokenizer(prompt)}')
print(f'"skunk" embeddings shape: {token_emb_layer(torch.tensor([42194], device=torch_device)).shape}')

In [ ]:
prompt = 'A picture of a puppy'

# Tokenize
text_input = tokenizer(
    prompt, 
    padding="max_length", 
    max_length=tokenizer.model_max_length, 
    truncation=True, 
    return_tensors="pt"
)
input_ids  = text_input.input_ids.to(torch_device)

# Get token embeddings
token_embeddings = token_emb_layer(input_ids)

# The new embedding. Which is now a mixture of the token embeddings for 'puppy' and 'skunk'
puppy_token_embedding       = token_emb_layer(torch.tensor(6829, device=torch_device))
skunk_token_embedding       = token_emb_layer(torch.tensor(42194, device=torch_device))
replacement_token_embedding = 0.5*puppy_token_embedding + 0.5*skunk_token_embedding

# Insert this into the token embeddings (
token_embeddings[0, torch.where(input_ids[0]==6829)] = replacement_token_embedding.to(torch_device)

# Combine token and position embeddings
input_embeddings = token_embeddings + position_embeddings

#  Feed input embedding through the transformer to get the output embeddings
modified_output_embeddings = get_output_embeds(input_embeddings)

# Generate an image with the obtained embeddings
generate_with_embs(modified_output_embeddings)

### Textual Inversion

Thus, we can feed the model with a modified token embedding, and use this to generate an image. We used the token embedding for "cat" in the previous example, but what if instead we learn a new token embedding for a specific concept? This is the idea behind 'textual inversion', in which a few example images are used to create a new token embedding.

![Overview image from the blog post](https://textual-inversion.github.io/static/images/training/training.JPG)
_Diagram from the [textual inversion blog post](https://textual-inversion.github.io/static/images/training/training.JPG), where the positional embeddings step is not shown for simplicity._

We will not cover how this training works, but we can try loading one of these new 'concepts' from the [community-created SD concepts library](https://huggingface.co/sd-concepts-library) and see how it fits in our example presented above. As an example, we will use https://huggingface.co/sd-concepts-library/birb-style. Download the learned_embeds.bin file from there and upload the file to wherever this notebook is before running the next cell.

In [ ]:
birb_embed = torch.load('/kaggle/input/datasets/OUR_KAGGLE_USER/birds-style-embeddings-for-sd/learned_embeds.bin')
birb_embed.keys(), birb_embed['<birb-style>'].shape

We get a dictionary with a key (the special placeholder we will use, <birb-style>) and the corresponding token embedding. As in the previous example, let us replace the "puppy" token embedding with this and see what we generate as images.

In [ ]:
prompt = 'A mouse in the style of puppy'

# Tokenize
text_input = tokenizer(
    prompt, 
    padding        = "max_length", 
    max_length     = tokenizer.model_max_length, 
    truncation     = True, 
    return_tensors = "pt"
)
input_ids  = text_input.input_ids.to(torch_device)

# Get the token embeddings
token_embeddings = token_emb_layer(input_ids)

# The new embedding for "a special birb" word
replacement_token_embedding = birb_embed['<birb-style>'].to(torch_device)

# Insert this embedding into our prompt's token embeddings
# in the place of the "puppy" token embedding
token_embeddings[0, torch.where(input_ids[0]==6829)] = replacement_token_embedding.to(torch_device)

# Combine token with position embeddings
input_embeddings = token_embeddings + position_embeddings

# Feed the altered input embeddings into the transformer to get the output embeddings
modified_output_embeddings = get_output_embeds(input_embeddings)

# And generate an image with these embeddings
generate_with_embs(modified_output_embeddings)

The token for "puppy" was replaced with one that captures a particular style of painting, but it could just as easily represent a specific object or class of objects.

There is as [inference notebook ](https://colab.research.google.com/github/huggingface/notebooks/blob/main/diffusers/stable_conceptualizer_inference.ipynb) from Hugging Face that makes it easy to use the different concepts, which properly handles using the names in prompts, such as "A \<cat-toy> in the style of \<birb-style>", without worrying about all the manual stuff we did just right now.

## Further Manipulation of the Embeddings

Besides just replacing the token embedding of a single word, there are various other tricks we can try. For example, what if we create a 'chimera' by averaging the embeddings of two different prompts?

In [ ]:
# Tokenize the first prompt
text_input1 = tokenizer(
    ["A mouse"], 
    padding        = "max_length", 
    max_length     = tokenizer.model_max_length, 
    truncation     = True, 
    return_tensors = "pt"
)
# Tokenize the second prompt
text_input2 = tokenizer(
    ["A leopard"], 
    padding        = "max_length", 
    max_length     = tokenizer.model_max_length, 
    truncation     = True, 
    return_tensors = "pt"
)
# Obtain embeddings for the first and second prompts
with torch.no_grad():
    text_embeddings1 = text_encoder(text_input1.input_ids.to(torch_device))[0]
    text_embeddings2 = text_encoder(text_input2.input_ids.to(torch_device))[0]

# Mix both embeddings together
mix_factor       = 0.35
mixed_embeddings = (text_embeddings1*mix_factor + \
                   text_embeddings2*(1-mix_factor))

# Generate an image for the mixed embeddings
generate_with_embs(mixed_embeddings)

## The UNET and CFG

Now it is time to focus on the diffusion model. This is typically a U-Net that takes in the noisy latents (x) and predicts the noise to remove. We use a conditional model that also takes in the timestep (t) and our text embedding (`encoder_hidden_states`) as conditioning. Feeding all of these into the model looks like this:
`noise_pred = unet(latents, t, encoder_hidden_states=text_embeddings)["sample"]`

We can try it out and see what the output looks like.

In [ ]:
# Prepate the noise scheduler
set_timesteps(scheduler, num_inference_steps)

# Get the schedule times and noise variances
t     = scheduler.timesteps[0]
sigma = scheduler.sigmas[0]

# Generate a random noise latent
latents = torch.randn(
  (batch_size, unet.in_channels, height // 8, width // 8),
  generator=generator,
)
latents = latents.to(torch_device)
# Scale the latents using the noise variances
latents = latents * scheduler.init_noise_sigma

# Tokenize the prompt "A macaw"
text_input = tokenizer(
    ['A macaw'], 
    padding        = "max_length", 
    max_length     = tokenizer.model_max_length, 
    truncation     = True, 
    return_tensors = "pt"
)
# Get the prompt embeddings
with torch.no_grad():
    text_embeddings = text_encoder(text_input.input_ids.to(torch_device))[0]

# Apply the latents, the diffusion times, and the prompt embeddings 
# into the U-Net to predict the noise that must be removed from the latents
with torch.no_grad():
    noise_pred = unet(latents, t, encoder_hidden_states=text_embeddings)["sample"]

# The U-Net preditions have the same shape as the input latents
print(f'Input latents   shape: {latents.shape}')
print(f'UNet preditions shape: {noise_pred.shape}')

Given a set of noisy latents, the model predicts the noise component. We can remove this noise from the noisy latents to see what the output image looks like (`latents_x0 = latents - sigma * noise_pred`). And we can remove most of the noise in this predicted output to get the (slightly less noisy hopefully) input for the next diffusion step. To visualize this let's generate another image, saving both the predicted output (x0) and the next step (xt-1) after every step:

In [ ]:
prompt              = 'Oil painting of an otter in a top hat'
height              = 512
width               = 512
num_inference_steps = 50
guidance_scale      = 8
generator           = torch.manual_seed(32)
batch_size          = 1

# Create a folder to store the results
!rm -rf steps/
!mkdir -p steps/

# Convert the conditioning text prompt into tokens using the CLIP tokenizer
text_input = tokenizer(
    [prompt], 
    padding        = "max_length", 
    max_length     = tokenizer.model_max_length, 
    truncation     = True, 
    return_tensors = "pt"
)

# Obtain the conditioning text prompt embeddings using the CLIP text encoder
with torch.no_grad():
    text_embeddings = text_encoder(text_input.input_ids.to(torch_device))[0]

# Convert an "empty" unconditioning text prompt into tokens using 
# the CLIP tokenizer (this is necessary in classifier free guidance - CFG)
max_length   = text_input.input_ids.shape[-1]
uncond_input = tokenizer(
    [""] * batch_size,
    padding        = "max_length",
    max_length     = max_length,
    return_tensors = "pt"
)

# Obtain the unconditioning text prompt embeddings using the CLIP text encoder
with torch.no_grad():
    uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))[0]

# Concatenate conditioning and unconditioning text prompt embeddings
text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

# Prepare the noise scheduler
set_timesteps(scheduler, num_inference_steps)

# Prepare the initial latents of the denoising process
latents = torch.randn(
  (batch_size, unet.in_channels, height // 8, width // 8),
  generator=generator,
)
# Scale the latents using the noise variance
latents = latents.to(torch_device)
latents = latents * scheduler.init_noise_sigma

# ...............................................................................
# Sampling/denoising loop to generate an image conditioned by the provided prompt
# ...............................................................................

for i, t in tqdm(enumerate(scheduler.timesteps), total=len(scheduler.timesteps)):
    # Duplicate the latents if we are applying classifier-free guidance 
    # to avoid executing two forward passes
    latent_model_input = torch.cat([latents] * 2)

    # Scale the latents (preconditioning)
    sigma = scheduler.sigmas[i]
    latent_model_input = scheduler.scale_model_input(latent_model_input, t)

    # Predict the noise that needs to be removed from the latents
    with torch.no_grad():
        noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embeddings)["sample"]

    # Apply CFG guidance
    noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
    noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

    # Get the predicted z_0, using our own calculations
    # latents_0 = latents - sigma * noise_pred
    scheduler_step = scheduler.step(noise_pred, t, latents)
    latents_0      = scheduler_step.pred_original_sample 

    # Obtain the previous noisy sample: given z_t predict z_{t-1}
    latents = scheduler_step.prev_sample

    # Convert the predicted z_0 and z_{t-1} to PIL images
    im_t0   = decode_latents_to_images(latents_0)[0]
    im_next = decode_latents_to_images(latents)[0]

    # Combine the two images and save them for later visualization
    im = Image.new('RGB', (1024, 512))
    im.paste(im_next, (0, 0))
    im.paste(im_t0, (512, 0))
    im.save(f'steps/{i:04}.jpeg')

In [ ]:
# With the images generated in every sampling step, we can make and show 
# a progress video (change the width to 1024 for full resolution)
!ffmpeg -v 1 -y -f image2 -framerate 12 -i steps/%04d.jpeg -c:v libx264 -preset slow -qp 18 -pix_fmt yuv420p out.mp4
mp4      = open('out.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=600 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

The content on the right is the final output (x_0) predicted at each step, and this is what is usually used to make progress animations. The content on the left is the image (x_{t-1}) corresponding to the 'next step' latent (z_{t-1}). It is interesting to compare the two, since watching the progress videos only one can think that drastic changes are happening especially at the early steps of the denoising, but since the changes made per-step are relatively small the actual process is much more gradual.



### Classifier Free Guidance

By default, the model does not often do what we ask. If we want it to follow the prompt better, we use classifier-free guidance (CFG). There is a good explanation in this video at the "AI coffee break GLIDE".

In our code, this is implemented with:

`noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)`

This works suprisingly well. Explore changing the `guidance_scale` in the code above and see how this affects the results. How high can we push it before the results get worse?

## Sampling

There are some complexity associated with `latents = scheduler.step(noise_pred, i, latents)["prev_sample"]`. How exactly does the sampler go from the current noisy latents to a slightly less noisy version? Why do not we just use the model in a single step? Are there other ways to view this?

The model tries to predict the noise in an image. For low noise values, we assume it does a pretty good job. For higher noise levels, it has a hard task! So instead of producing a perfect image in a single step, where the results tend to be a blurry image (as illustrated in the beginning of the video produced above), samplers use the model to make predictions that move in small steps towards the model prediction (removing some of the noise at each step) and then get another prediction based on the marginally-less-noisy latents, and hope that this iteratively improves the latents.

Different samplers do this in different ways. We can inspect the implementation of the default LMS sampler by running the next cell.

In [ ]:
??scheduler.step

# Guidance

As a final example, we are going to see how we can add more control to the generation process. At each step, we are going to use our model as before to predict the noise component of x. Then, we will use this to produce a predicted output image, and apply some loss function to this image.

This function can be anything, but let us demonstrate with a super simple example. If we want images that have a lot of blue, we can craft a loss function that gives a high loss if pixels have a low blue component.

In [ ]:
def blue_loss(images):
    # How far are the blue channel values from 0.9
    error = torch.abs(images[:,2] - 0.9).mean() # [:,2] means all images in the batch, only the blue channel
    return error

During each update step, we find the gradient of the loss with respect to the current noisy latents, and tweak them in the direction that reduces this loss as well as performing the normal update step.

In [ ]:
prompt              = 'A campfire (oil on canvas)' #@param
height              = 512                   # default image height in Stable Diffusion
width               = 512                   # default image width of Stable Diffusion
num_inference_steps = 50  #@param           # number of denoising steps
guidance_scale      = 8   #@param           # scale applied in classifier-free guidance
generator           = torch.manual_seed(32) # seed generator to create the inital latent noise
batch_size          = 1
blue_loss_scale     = 200 #@param

# Convert the conditioning text prompt into tokens using the CLIP tokenizer
text_input = tokenizer(
    [prompt], 
    padding        = "max_length", 
    max_length     = tokenizer.model_max_length, 
    truncation     = True, 
    return_tensors = "pt"
)

# Obtain the conditioning text prompt embeddings using the CLIP text encoder
with torch.no_grad():
    text_embeddings = text_encoder(text_input.input_ids.to(torch_device))[0]

# Convert an "empty" unconditioning text prompt into tokens using 
# the CLIP tokenizer (this is necessary in classifier free guidance - CFG)
max_length = text_input.input_ids.shape[-1]
uncond_input = tokenizer(
    [""] * batch_size, 
    padding        = "max_length", 
    max_length     = max_length, 
    return_tensors = "pt",
)

# Obtain the unconditioning text prompt embeddings using the CLIP text encoder
with torch.no_grad():
    uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))[0]

# Concatenate conditioning and unconditioning text prompt embeddings
text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

# Prepare the noise scheduler
set_timesteps(scheduler, num_inference_steps)

# Prepare the initial latents of the denoising process
latents = torch.randn(
  (batch_size, unet.in_channels, height // 8, width // 8),
  generator=generator,
)
# Scale the latents using the noise variance
latents = latents.to(torch_device)
latents = latents * scheduler.init_noise_sigma

# ...............................................................................
# Sampling/denoising loop to generate an image conditioned by the provided prompt
# ...............................................................................

for i, t in tqdm(enumerate(scheduler.timesteps), total=len(scheduler.timesteps)):
    # Duplicate the latents if we are applying classifier-free guidance 
    # to avoid executing two forward passes
    latent_model_input = torch.cat([latents] * 2)
    # Scale the latents (preconditioning)
    sigma              = scheduler.sigmas[i]
    latent_model_input = scheduler.scale_model_input(latent_model_input, t)

    # Predict the noise that needs to be removed from the latents
    with torch.no_grad():
        noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embeddings)["sample"]

    # Apply CFG guidance
    noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
    noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

    # APPLY ADDITIONAL GUIDANCE .................................................
    if i%5 == 0:
        # Specify that the gradients are applied to the latents
        latents = latents.detach().requires_grad_()

        # Get the predicted z_0
        latents_0 = latents - sigma * noise_pred
        # latents_0 = scheduler.step(noise_pred, t, latents).pred_original_sample

        # Decode predicted z_0 to the image space (x_0)
        denoised_images = vae.decode((1 / 0.18215) * latents_0).sample / 2 + 0.5 # range (0, 1)

        # Calculate the loss as the error of the image blue channel values relative to 0.9
        loss = blue_loss(denoised_images) * blue_loss_scale

        # print the loss every 5 steps
        if i%5==0:
            print(i, 'loss:', loss.item())

        # Get the gradients of the loss with respect to the latents
        cond_grad = torch.autograd.grad(loss, latents)[0]

        # Update the latents using the gradients
        latents = latents.detach() - cond_grad * sigma**2

    # Execute a scheduler step to get a new latent prediction (z_{t-1})
    latents = scheduler.step(noise_pred, t, latents).prev_sample


decode_latents_to_images(latents)[0]

If we alter the `blue_loss_scale`, low values correspond to images mostly colored in red and orange thanks to the prompt, and high values correspond to mostly bluish images. Too high scales and we get a complete blue image.

Since this process is slow, we only applied the additional guidance provide the loss gradients once every 5 iterations. One may consider applying a low `blue_loss_scale` value to the loss and calculating the loss at every iteration.

NOTE: We should set latents `requires_grad=True` **before** we do the forward pass of the UNet (removing `with torch.no_grad()`) if we want mode accurate gradients. But this requires a lot more memory.

Guiding with classifier models can give you images of a specific class. Guiding with a model like CLIP can help better match a text prompt. Guiding with a style loss can help add a particular style. Guiding with some sort of perceptual loss can force it towards the overall look of a target image. 

# Conclusions

This notebook tries to clarify a few aspects of conditional image generation with Stable Diffusion, namely (i) the text conditioning manipulation, (ii) the noise scheduler role, (iii) the sampling loop, (iv) the classifier-free guidance, and (v) using a customized loss's gradients to reinforce the guidance in a specific direction.


In [ ]:
!zip -r sd_sampling_animation.zip /kaggle/working
from IPython.display import FileLink
FileLink(r'sd_sampling_animation.zip')